In [1]:
from pathlib import Path
print(Path("../data/ats_knowledge.json").exists()) 

True


In [2]:
import json
from pathlib import Path
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

C:\Users\Owner\AppData\Local\Temp\ipykernel_13240\2381520303.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

In [3]:
data_path = Path("../data/ats_knowledge.json")

with open(data_path) as f:
    raw_docs = json.load(f)

print(f"Loaded {len(raw_docs)} documents")
print(json.dumps(raw_docs[0], indent=2))

Loaded 50 documents
{
  "title": "Single column layout vs multi-column for ATS compatibility",
  "category": "formatting",
  "content": "When creating a resume, it's essential to consider ATS compatibility to ensure your document can be accurately parsed and processed. In terms of layout, a single-column format is highly recommended. This is because multi-column layouts can cause issues with ATS parsing, as the system may struggle to correctly identify and extract relevant information.\n\nFor example, if you have a multi-column layout with your work experience in one column and your skills in another, the ATS may incorrectly merge the two columns, resulting in a jumbled and unreadable format. In contrast, a single-column layout allows the ATS to easily follow the flow of your resume and extract the necessary information.\n\nTo optimize your resume for ATS compatibility, use a single-column layout with clear headings, bullet points, and white space to separate sections. Avoid using tabl

In [4]:
# Cell 3 — convert to LangChain Documents
documents = [
    Document(
        page_content=f"{doc['title']}\n\n{doc['content']}",
        metadata={"category": doc["category"], "title": doc["title"]}
    )
    for doc in raw_docs
]

print(f"Created {len(documents)} Document objects")
print(documents[0].page_content[:300])
print("Metadata:", documents[0].metadata)

Created 50 Document objects
Single column layout vs multi-column for ATS compatibility

When creating a resume, it's essential to consider ATS compatibility to ensure your document can be accurately parsed and processed. In terms of layout, a single-column format is highly recommended. This is because multi-column layouts can 
Metadata: {'category': 'formatting', 'title': 'Single column layout vs multi-column for ATS compatibility'}


In [5]:
# Cell 4 — build FAISS index (makes one embedding API call for all 50 docs)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(documents, embeddings)

print("FAISS index built")
print(f"Number of vectors: {vectorstore.index.ntotal}")

FAISS index built
Number of vectors: 50


In [6]:
# Cell 5 — test retrieval
query = "how to add missing cloud skills like AWS to my resume"

results = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} [{doc.metadata['category']}] ---")
    print(doc.metadata['title'])
    print(doc.page_content[:200])
    print()

--- Result 1 [keywords] ---
Industry-specific keywords for data and technology roles
Industry-specific keywords for data and technology roles

When optimizing resumes for data and technology roles, incorporating industry-specific keywords is crucial. For data roles, keywords such as d

--- Result 2 [keywords] ---
Technical skills section optimisation for ATS
Technical skills section optimisation for ATS

To optimize the technical skills section for ATS, use a clear and concise format, listing skills in a bullet-point or table format. Use specific keywords

--- Result 3 [keywords] ---
Hard skills vs soft skills in ATS keyword matching
Hard skills vs soft skills in ATS keyword matching

When optimizing a resume for ATS keyword matching, it's essential to strike a balance between hard skills and soft skills. Hard skills, such as prog

--- Result 4 [tailoring] ---
Career changers - how to frame transferable skills for ATS
Career changers - how to frame transferable skills for ATS

When tra

In [7]:
# Cell 6 — retrieval with scores
query = "missing keywords in skills section"

results = vectorstore.similarity_search_with_score(query, k=5)

for doc, score in results:
    print(f"[{score:.4f}] {doc.metadata['title']}")

[0.9742] Technical skills section optimisation for ATS
[1.0408] Where to place keywords for maximum ATS impact
[1.0652] Hard skills vs soft skills in ATS keyword matching
[1.0682] Unusual section headings that ATS cannot categorise
[1.1026] Required vs nice-to-have skills - how to prioritise


In [8]:
vectorstore.save_local("../data/faiss_index")
print("Index saved")

Index saved


In [9]:
# Cell 8 — reload and verify
loaded = FAISS.load_local(
    "../data/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print(f"Reloaded {loaded.index.ntotal} vectors")

results = loaded.similarity_search("ATS keyword optimization", k=3)
for doc in results:
    print("-", doc.metadata['title'])

Reloaded 50 vectors
- Where to place keywords for maximum ATS impact
- Keyword stuffing - why it backfires in modern ATS systems
- How ATS keyword matching works - exact vs semantic matching
